# 🛡️ Face Anti-Spoofing — 03: CelebA-Spoof 서브셋 다운로드
> AI Security & Application · 단국대학교 소프트웨어학과  
> 학번: 32214391 · 조현수

---
## 체크리스트
- [ ] Cell 1: Google Drive 마운트 + 경로 고정
- [ ] Cell 2: kaggle.json Drive 저장 (최초 1회만)
- [ ] Cell 3: Kaggle 인증
- [ ] Cell 4: train_label.json Drive로 다운로드
- [ ] Cell 5: Spoof Type별 분포 확인
- [ ] Cell 6: 6만장 균형 샘플링
- [ ] Cell 7: 100장 테스트 다운로드
- [ ] Cell 8: 전체 다운로드 실행

In [18]:
# ── Cell 1: Google Drive 마운트 + 경로 고정 ──────────────────
# ⚠️ 팝업이 뜨면 '허용'을 눌러주세요.
# 이미 마운트된 상태면 'Already mounted' 메시지가 뜨고 그냥 넘어갑니다.

from google.colab import drive
drive.mount('/content/drive')

import os

# 모든 노트북에서 공통으로 쓸 경로 — 여기만 바꾸면 전체 적용
BASE = '/content/drive/MyDrive/face-anti-spoofing'

# 필요한 폴더 한번에 생성
dirs = [
    f'{BASE}/data',
    f'{BASE}/data/subset/live',
    f'{BASE}/data/subset/print',
    f'{BASE}/data/subset/replay',
    f'{BASE}/data/subset/mask',
    f'{BASE}/samples/live',
    f'{BASE}/samples/print',
    f'{BASE}/samples/replay',
    f'{BASE}/samples/toy',
    f'{BASE}/reports',
    f'{BASE}/models',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print(f'✅ Drive 마운트 완료')
print(f'✅ BASE = {BASE}')
print(f'✅ 폴더 구조 생성 완료')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive 마운트 완료
✅ BASE = /content/drive/MyDrive/face-anti-spoofing
✅ 폴더 구조 생성 완료


In [21]:
import os

KAGGLE_IN_DRIVE = f'{BASE}/kaggle.json'
print(os.path.exists(KAGGLE_IN_DRIVE))  # True 뜨면 성공

True


In [22]:
# ── Cell 3: Kaggle 인증 ───────────────────────────────────────
# Drive에 저장된 kaggle.json을 읽어서 인증
# 런타임 재시작 후에도 Drive에서 읽으므로 재업로드 불필요

import shutil

KAGGLE_SRC = f'{BASE}/kaggle.json'
KAGGLE_DST = os.path.expanduser('~/.kaggle/kaggle.json')

os.makedirs(os.path.dirname(KAGGLE_DST), exist_ok=True)
shutil.copy(KAGGLE_SRC, KAGGLE_DST)
os.chmod(KAGGLE_DST, 0o600)

# 인증 확인
result = os.popen('kaggle datasets list --max-size 1 2>&1').read()
if 'Error' in result or 'Invalid' in result:
    print(f'❌ Kaggle 인증 실패:\n{result}')
else:
    print('✅ Kaggle 인증 완료')

✅ Kaggle 인증 완료


In [23]:
# ── Cell 4: train_label.json Drive로 다운로드 ─────────────────
# 이미 Drive에 있으면 스킵 (재실행 안전)

import json

LABEL_PATH = f'{BASE}/data/train_label.json'
TARGET_JSON = 'CelebA_Spoof/metas/intra_test/train_label.json'

if os.path.exists(LABEL_PATH):
    print(f'✅ 이미 존재: {LABEL_PATH}')
    print('다운로드 스킵')
else:
    print('📥 train_label.json 다운로드 중...')
    os.system(
        f'kaggle datasets download -d mabdullahsajid/celeba-spoofing '
        f'-f "{TARGET_JSON}" -p /content/tmp_label'
    )
    # 압축 해제
    os.system('unzip -o /content/tmp_label/train_label.json.zip -d /content/tmp_label')
    # Drive로 이동
    extracted = '/content/tmp_label/train_label.json'
    if os.path.exists(extracted):
        shutil.copy(extracted, LABEL_PATH)
        print(f'✅ Drive 저장 완료: {LABEL_PATH}')
    else:
        print('❌ 압축 해제 실패. 경로를 확인하세요.')

# 파싱 확인
with open(LABEL_PATH, 'r') as f:
    meta = json.load(f)

first_key = list(meta.keys())[0]
print(f'\n📍 첫 번째 키: {first_key}')
print(f'📊 전체 샘플 수: {len(meta):,}장')
print(f'🎯 index 40 (Spoof Type): {meta[first_key][40]}')
print(f'💡 index 41 (Illumination): {meta[first_key][41]}')
print(f'🌍 index 42 (Environment): {meta[first_key][42]}')

📥 train_label.json 다운로드 중...
✅ Drive 저장 완료: /content/drive/MyDrive/face-anti-spoofing/data/train_label.json

📍 첫 번째 키: Data/train/2623/live/000000.jpg
📊 전체 샘플 수: 494,405장
🎯 index 40 (Spoof Type): 0
💡 index 41 (Illumination): 0
🌍 index 42 (Environment): 0


In [24]:
# ── Cell 5: Spoof Type별 분포 확인 ───────────────────────────

from collections import defaultdict

TYPE_LABEL = {
    0:  'Live',
    1:  'Print (색상지)',
    2:  'Print (흑백지)',
    3:  'Replay (화면)',
    4:  'Paper Cut',
    5:  'Replay (태블릿)',
    6:  'Print (구겨진)',
    7:  'Replay (스마트폰)',
    8:  'Print (부분)',
    9:  'Paper Cut (눈구멍)',
    10: '3D Mask (Toy)',
}

type_to_paths = defaultdict(list)
for img_path, annot in meta.items():
    spoof_type = annot[40]
    type_to_paths[spoof_type].append(img_path)

print(f'{'Type':>6}  {'라벨':<20}  {'장수':>8}')
print('-' * 42)
for t in sorted(type_to_paths.keys()):
    label = TYPE_LABEL.get(t, '?')
    count = len(type_to_paths[t])
    print(f'{t:>6}  {label:<20}  {count:>8,}')
print('-' * 42)
print(f'{'합계':>28}  {sum(len(v) for v in type_to_paths.values()):>8,}')

  Type  라벨                          장수
------------------------------------------
     0  Live                   162,462
     1  Print (색상지)             35,547
     2  Print (흑백지)             31,221
     3  Replay (화면)             31,776
     4  Paper Cut               33,647
     5  Replay (태블릿)            30,167
     6  Print (구겨진)             33,285
     7  Replay (스마트폰)           31,072
     8  Print (부분)              33,085
     9  Paper Cut (눈구멍)         31,527
    10  3D Mask (Toy)           40,616
------------------------------------------
                          합계   494,405


In [25]:
# ── Cell 6: 6만장 균형 샘플링 ────────────────────────────────
# Live 15k / Print 15k / Replay 15k / Mask 15k = 60k

import random
random.seed(42)

TARGET = 15000

# 카테고리별 타입 그룹
GROUPS = {
    'live':   [0],
    'print':  [1, 2, 6, 8],
    'replay': [3, 5, 7],
    'mask':   [4, 9, 10],
}

selected = {}
for cat, type_nums in GROUPS.items():
    pool = []
    for t in type_nums:
        pool.extend(type_to_paths[t])
    n = min(TARGET, len(pool))
    selected[cat] = random.sample(pool, n)

total = sum(len(v) for v in selected.values())
print(f'✅ 샘플링 완료 — 총 {total:,}장')
print()
for cat, paths in selected.items():
    print(f'  {cat:<8}: {len(paths):>6,}장')

# 경로 목록 Drive에 저장 (다운로드 재개 가능하도록)
LIST_PATH = f'{BASE}/data/download_list.txt'
with open(LIST_PATH, 'w') as f:
    for cat, paths in selected.items():
        for p in paths:
            f.write(f'{cat}\t{p}\n')

print(f'\n✅ 경로 목록 저장: {LIST_PATH}')

✅ 샘플링 완료 — 총 60,000장

  live    : 15,000장
  print   : 15,000장
  replay  : 15,000장
  mask    : 15,000장

✅ 경로 목록 저장: /content/drive/MyDrive/face-anti-spoofing/data/download_list.txt


In [26]:
# ── Cell 7: 100장 테스트 다운로드 ────────────────────────────
# 전체 실행 전에 25장×4카테고리로 파이프라인 검증

import glob
import shutil

TEST_N = 25  # 카테고리당 테스트 장수
DATASET = 'mabdullahsajid/celeba-spoofing'

def download_image(img_path, save_dir, tmp_dir='/content/tmp_dl'):
    """단일 이미지 다운로드 → save_dir로 이동"""
    filename = img_path.split('/')[-1]
    save_path = os.path.join(save_dir, filename)

    # 이미 있으면 스킵
    if os.path.exists(save_path):
        return True

    os.makedirs(tmp_dir, exist_ok=True)
    kaggle_path = f'CelebA_Spoof/{img_path}'

    ret = os.system(
        f'kaggle datasets download -d {DATASET} '
        f'-f "{kaggle_path}" -p "{tmp_dir}" -q'
    )

    # 압축 해제
    for z in glob.glob(os.path.join(tmp_dir, '*.zip')):
        os.system(f'unzip -q -o "{z}" -d "{tmp_dir}"')
        os.remove(z)

    # 파일 찾아서 이동
    found = glob.glob(os.path.join(tmp_dir, '**', filename), recursive=True)
    if found:
        shutil.copy(found[0], save_path)
        shutil.rmtree(tmp_dir, ignore_errors=True)
        return True
    return False

# 테스트 실행
print('🧪 100장 테스트 다운로드 시작...\n')
success, fail = 0, 0

for cat, paths in selected.items():
    save_dir = f'{BASE}/data/subset/{cat}'
    test_paths = paths[:TEST_N]

    for i, p in enumerate(test_paths):
        ok = download_image(p, save_dir)
        if ok:
            success += 1
        else:
            fail += 1
        print(f'  [{cat}] {i+1}/{TEST_N} — {"✅" if ok else "❌"} {p.split("/")[-1]}')

print(f'\n✅ 성공: {success}장 / ❌ 실패: {fail}장')

# 저장 확인
for cat in selected.keys():
    n = len(os.listdir(f'{BASE}/data/subset/{cat}'))
    print(f'  Drive/subset/{cat}: {n}장')

🧪 100장 테스트 다운로드 시작...

  [live] 1/25 — ✅ 088339.jpg
  [live] 2/25 — ✅ 020005.jpg
  [live] 3/25 — ✅ 219271.jpg
  [live] 4/25 — ✅ 195383.jpg
  [live] 5/25 — ✅ 178065.jpg
  [live] 6/25 — ✅ 111055.jpg
  [live] 7/25 — ✅ 081315.jpg
  [live] 8/25 — ✅ 435159.jpg
  [live] 9/25 — ✅ 069020.jpg
  [live] 10/25 — ✅ 471015.jpg
  [live] 11/25 — ✅ 336827.jpg
  [live] 12/25 — ✅ 025505.jpg
  [live] 13/25 — ✅ 023965.jpg
  [live] 14/25 — ✅ 074487.jpg
  [live] 15/25 — ✅ 174406.jpg
  [live] 16/25 — ✅ 185675.jpg
  [live] 17/25 — ✅ 403267.jpg
  [live] 18/25 — ✅ 480310.jpg
  [live] 19/25 — ✅ 021284.jpg
  [live] 20/25 — ✅ 447835.jpg
  [live] 21/25 — ✅ 158570.jpg
  [live] 22/25 — ✅ 434784.jpg
  [live] 23/25 — ✅ 334904.jpg
  [live] 24/25 — ✅ 175846.jpg
  [live] 25/25 — ✅ 358615.jpg
  [print] 1/25 — ✅ 390397.jpg
  [print] 2/25 — ✅ 445134.jpg
  [print] 3/25 — ✅ 269274.jpg
  [print] 4/25 — ✅ 171641.jpg
  [print] 5/25 — ✅ 462520.jpg
  [print] 6/25 — ✅ 233819.jpg
  [print] 7/25 — ✅ 076990.jpg
  [print] 8/25 — ✅ 421068.

In [27]:
# Cell 8 대안 — 중간발표용 미니 서브셋 (카테고리당 1,500장 = 총 6,000장)
MINI_TARGET = 1500

for cat, paths in selected.items():
    save_dir = f'{BASE}/data/subset/{cat}'
    already = len(os.listdir(save_dir))
    remaining = paths[already:MINI_TARGET]

    print(f'\n📂 [{cat}] {already}장 완료 → {len(remaining)}장 추가')

    for i, p in enumerate(remaining):
        ok = download_image(p, save_dir)
        if (i+1) % 100 == 0:
            print(f'  {i+1}/{len(remaining)} 완료')

print('\n✅ 중간발표용 서브셋 완료')


📂 [live] 25장 완료 → 1475장 추가
  100/1475 완료
  200/1475 완료
  300/1475 완료
  400/1475 완료
  500/1475 완료
  600/1475 완료
  700/1475 완료
  800/1475 완료
  900/1475 완료
  1000/1475 완료
  1100/1475 완료
  1200/1475 완료
  1300/1475 완료
  1400/1475 완료

📂 [print] 25장 완료 → 1475장 추가
  100/1475 완료
  200/1475 완료
  300/1475 완료
  400/1475 완료
  500/1475 완료
  600/1475 완료
  700/1475 완료
  800/1475 완료
  900/1475 완료
  1000/1475 완료
  1100/1475 완료
  1200/1475 완료
  1300/1475 완료
  1400/1475 완료

📂 [replay] 25장 완료 → 1475장 추가
  100/1475 완료
  200/1475 완료
  300/1475 완료
  400/1475 완료
  500/1475 완료
  600/1475 완료
  700/1475 완료
  800/1475 완료
  900/1475 완료
  1000/1475 완료
  1100/1475 완료
  1200/1475 완료
  1300/1475 완료
  1400/1475 완료

📂 [mask] 25장 완료 → 1475장 추가
  100/1475 완료
  200/1475 완료
  300/1475 완료
  400/1475 완료
  500/1475 완료
  600/1475 완료
  700/1475 완료
  800/1475 완료
  900/1475 완료
  1000/1475 완료
  1100/1475 완료
  1200/1475 완료
  1300/1475 완료
  1400/1475 완료

✅ 중간발표용 서브셋 완료


## ✅ 완료 체크리스트

| 항목 | 상태 |
|------|------|
| Google Drive 마운트 + 경로 고정 | ⬜ |
| kaggle.json Drive 저장 (최초 1회) | ⬜ |
| Kaggle 인증 | ⬜ |
| train_label.json 확보 | ⬜ |
| Spoof Type 분포 확인 | ⬜ |
| 6만장 샘플링 목록 생성 | ⬜ |
| 100장 테스트 다운로드 성공 | ⬜ |
| 전체 6만장 다운로드 완료 | ⬜ |

### 다음 단계
- **04_preprocess_train.ipynb:** BB.txt 크롭 + 멀티태스크 MobileNetV2 학습